# RVC v2 48k 训练 - Kaggle

设置 Accelerator 为任一 GPU（推荐 T4 x2），Internet 设为 ON，然后运行全部 Cell。

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), '未检测到 CUDA GPU'
print(f'[RVC] 检测到 {torch.cuda.device_count()} 张 GPU')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'[RVC] cuda:{i} {props.name} ({props.total_memory / 2**30:.1f} GB)')
print('[RVC] 双卡将自动使用 DDP；单卡将自动使用 cuda:0')

In [ ]:
import re, subprocess
from kaggle_secrets import UserSecretsClient
repo_commit = UserSecretsClient().get_secret('RVC_REPO_COMMIT').strip()
assert re.fullmatch(r'[0-9a-fA-F]{40}', repo_commit), '请在 Kaggle Secret 设置 40 位 RVC_REPO_COMMIT'
root = '/kaggle/working/HRVC'
subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', 'https://github.com/lingrana/rvc_train_kaggle.git', root], check=True)
subprocess.run(['git', '-C', root, 'checkout', '--detach', repo_commit], check=True)
%cd /kaggle/working/HRVC

In [ ]:
import os, subprocess, sys
os.environ['PYTHONNOUSERSITE'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '.', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'protobuf==5.29.4', 'huggingface_hub==0.36.0', 'kagglehub==0.3.13', '-q'], check=True)
subprocess.run([sys.executable, 'tools/kaggle_bootstrap.py', '--project-root', '/kaggle/working/HRVC'], check=True)
print('[RVC] 依赖与标准 v2 48k 底模校验完成')

In [ ]:
import hashlib, hmac, os, stat, subprocess, urllib.request
binary = '/kaggle/working/HRVC/cloudflared-linux-amd64'
cloudflared_url = 'https://github.com/cloudflare/cloudflared/releases/download/2026.7.3/cloudflared-linux-amd64'
cloudflared_sha256 = '9d71c677db00134c1bd4144b7783486b654ad281b1ea62b4972098d19f770f17'
if not os.path.isfile(binary) or not hmac.compare_digest(hashlib.sha256(open(binary, 'rb').read()).hexdigest(), cloudflared_sha256):
    urllib.request.urlretrieve(cloudflared_url, binary)
actual = hashlib.sha256(open(binary, 'rb').read()).hexdigest()
assert hmac.compare_digest(actual, cloudflared_sha256), f'cloudflared SHA-256 校验失败: {actual}'
os.chmod(binary, os.stat(binary).st_mode | stat.S_IXUSR)
subprocess.run(['python', '-u', 'tools/kaggle_launch.py'], check=True)